In [8]:
import json
import gzip
import os
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from scipy.special import expit as sigmoid
from sklearn.metrics import f1_score
from skmultilearn.model_selection import iterative_train_test_split
import optuna

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)


In [2]:
from datasets import load_from_disk

dataset = load_from_disk("../binary_dataset")

train_dataset = dataset["train"]
dev_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [3]:
train_dataset

Dataset({
    features: ['u', 'id', 'ts', 'text', 'Turku_NLP', 'Turku_NLP_sub', 'Tags', 'web-register', 'Binary', 'source'],
    num_rows: 3577
})

In [7]:
print(train_dataset.features)
print(train_dataset[0]["Binary"])
print(type(train_dataset[0]["Binary"]))

{'u': Value('string'), 'id': Value('string'), 'ts': Value('string'), 'text': Value('string'), 'Turku_NLP': Value('string'), 'Turku_NLP_sub': Value('string'), 'Tags': Value('string'), 'web-register': Value('string'), 'Binary': ClassLabel(names=['0', '1']), 'source': Value('string')}
1
<class 'int'>


In [10]:
from collections import Counter

print(Counter(train_dataset["Binary"]))

Counter({1: 3163, 0: 414})


0 = junk, 1 = non-junk.

In [ ]:
# rename target to labels
train_dataset = train_dataset.rename_column("Binary", "labels")
dev_dataset = dev_dataset.rename_column("Binary", "labels")
test_dataset = test_dataset.rename_column("Binary", "labels")


In [ ]:
NUM_LABELS = 2

In [ ]:
MODEL_NAME = "BAAI/bge-m3-retromae"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=1024,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
for ds in (train_dataset, dev_dataset, test_dataset):
    ds.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"],
    )

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

Micro-F1 is effectively the same as accuracy for single-label classification

In [ ]:
from sklearn.metrics import recall_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

#Do not want to leave any Junk behind
    return {"recall_0": recall_score(labels, predictions, pos_label=0)} #recall for class 0

In [ ]:
# ------------------------------------------------
# Optuna objective
# ------------------------------------------------

#Bayesian optimization (sample hyperparameters intelligently across trials)
def objective(trial):

    learning_rate = trial.suggest_float("learning_rate", 5e-6, 3e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.15)

    per_device_batch = trial.suggest_categorical("batch_size", [4, 8])
    grad_accum = trial.suggest_categorical("grad_accum", [4, 8])

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        problem_type="single_label_classification",
    )

    args = TrainingArguments(
        output_dir=f"./optuna_ham_spam/trial_{trial.number}",
        overwrite_output_dir=True,

        num_train_epochs=10,
        per_device_train_batch_size=per_device_batch,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=grad_accum,

        learning_rate=learning_rate,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,

        eval_strategy="epoch",
        logging_strategy="epoch",

        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_recall_0",
        report_to="none",

        seed=42,
        bf16=True,
        greater_is_better=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()
    metrics = trainer.evaluate()

    return metrics["eval_recall_0"]

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# 1) Get the best checkpoint from your Optuna trial (example assumes you kept it)
# If you ran only objective() directly, you need to rerun best trial or store it.
# For illustration: load from some known best path
# --- run optuna ---
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=8)  # set n_trials

# --- load best checkpoint ---
best_ckpt = f"./optuna_ham_spam/trial_{study.best_trial.number}"
best_model = AutoModelForSequenceClassification.from_pretrained(best_ckpt)

# --- evaluate on test and print classification table ---
trainer = Trainer(
    model=best_model,
    args=TrainingArguments(output_dir="./tmp", report_to="none"),
    tokenizer=tokenizer,
)

print("\n===== BEST TRIAL =====")
print(study.best_trial.params)
print("Best Recall:", study.best_value)

pred = trainer.predict(test_dataset)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)

print("\n=============")
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


target_names = ["class_0", "class_1"]
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

df_cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred),
    index=target_names,
    columns=target_names
)
print(df_cm)

def predict_with_threshold(trainer, dataset, threshold):
    out = trainer.predict(dataset)
    logits = out.predictions
    if isinstance(logits, (tuple, list)):
        logits = logits[0]
    p0 = probs_class0_from_logits(logits)
    pred_labels = np.where(p0 >= threshold, 0, 1)  # 0=junk, 1=keep
    return pred_labels, p0


After `trainer.train()` (and Optuna finishes), do:

1) **Load the best trial checkpoint**
2) **Run `tune_threshold_recall0_under_keep90` on the dev set**
3) **Use the returned `threshold` to batch-predict and filter the big unlabeled corpus**

Here’s the concrete flow.

## Step 1: get the best checkpoint from Optuna
```python
# after study.optimize(...)
best_trial = study.best_trial
best_dir = f"./optuna_xlmr/trial_{best_trial.number}"
```

## Step 2: tune the threshold on dev using the best checkpoint
Create a new trainer with the best model (so predict uses that model), then tune:

```python
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

best_model = AutoModelForSequenceClassification.from_pretrained(
    best_dir, num_labels=2
)

# tokenizer is the same as before
inf_args = TrainingArguments(
    output_dir="./tmp_eval",
    per_device_eval_batch_size=16,
    bf16=True,
    report_to="none",
)

best_trainer = Trainer(
    model=best_model,
    args=inf_args,
    tokenizer=tokenizer,
)

best = tune_threshold_recall0_under_keep90(
    trainer=best_trainer,
    dev_dataset=dev_dataset,   # YOUR already-tokenized dev_dataset w/ labels
    thresholds=np.linspace(0.0, 1.0, 201),
    min_recall1=0.90
)

best_threshold = best["threshold"]
print("Best threshold:", best_threshold)
```

## Step 3: apply to big unlabeled corpus in batches
You don’t need to manually batch if you use `trainer.predict()` (it batches internally). You just need to:
- tokenize the unlabeled corpus
- set the format to `input_ids`, `attention_mask`
- call `trainer.predict()`
- classify with the tuned `threshold`
- write results to disk

### Example with an HF Dataset named `big_corpus` with column `"text"`
```python
# 1) tokenize
big_tok = big_corpus.map(
    lambda batch: tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=1024,
    ),
    batched=True
)

big_tok.set_format(type="torch", columns=["input_ids", "attention_mask"])

# 2) predict
out = best_trainer.predict(big_tok)
logits = out.predictions
if isinstance(logits, (tuple, list)):
    logits = logits[0]

p0 = probs_class0_from_logits(logits)  # P(junk)
pred_labels = np.where(p0 >= best_threshold, 0, 1)  # 0=junk, 1=keep

# 3) save filtering results
junk_mask = (pred_labels == 0)
big_tok = big_tok.remove_columns([c for c in big_tok.column_names if c not in ["text"]])  # optional

# simplest: save indices
junk_indices = np.nonzero(junk_mask)[0]
np.save("junk_indices.npy", junk_indices)
```

If you tell me what form your “big unlabeled corpus” is (HF `Dataset`? JSONL files? list? column names?), I can adapt this so it works with streaming / writing JSONL instead of keeping everything in memory.

import numpy as np
from sklearn.metrics import recall_score

def probs_class0_from_logits(logits):
    # logits: (N,2)
    exp = np.exp(logits - logits.max(axis=-1, keepdims=True))
    probs = exp / exp.sum(axis=-1, keepdims=True)
    return probs[:, 0]

def tune_threshold_recall0_under_keep90(trainer, dev_dataset, thresholds=None, min_recall1=0.90):
    if thresholds is None:
        thresholds = np.linspace(0.0, 1.0, 201)

    out = trainer.predict(dev_dataset)
    logits = out.predictions
    if isinstance(logits, (tuple, list)):
        logits = logits[0]  # take first element if needed

    labels = out.label_ids  # 0/1

    p0 = probs_class0_from_logits(logits)  # P(class=0=junk)

    best = None
    for t in thresholds:
        pred = np.where(p0 >= t, 0, 1)  # predict junk=0 if confidence junk is high

        recall1 = recall_score(labels, pred, pos_label=1)
        if recall1 >= min_recall1:
            recall0 = recall_score(labels, pred, pos_label=0)
            if best is None or recall0 > best["recall_0"]:
                best = {"threshold": float(t), "recall_0": float(recall0), "recall_1": float(recall1)}

    return best
